# Снятие фикстур с fedstat.ru (ЕМИСС)

Ноутбук ходит на fedstat.ru, сохраняет **сырой HTML** страницы индикатора и **сырой ответ SDMX/Excel**, а также разобранную таблицу фильтров и метаданные. Эти файлы нужны для разработки Python-библиотеки.

**Порядок работы:**
1. Запусти ячейку с настройками и ячейку импорта.
2. Сначала **`dry-run`** (только осмотр: сохранит HTML, покажет фильтры и наличие CSRF — данные не качает).
3. Если фильтры распарсились и CSRF найден — запусти **захват данных**.
4. Запакуй `fixtures/` в zip и пришли мне.

Логика парсинга (`parse_js1/js2`, CSRF, POST-тело) — порт из R-пакета `fedstatAPIr`, лежит в `fedstat_capture.py`.

## 1. Настройки

In [1]:
# Список indicator_id (из URL вида https://www.fedstat.ru/indicator/31074)
INDICATORS = ["31452"]          # можно несколько: ["31074", "37426"]

OUT_DIR   = "fixtures"          # куда складывать файлы
FORMATS   = ["sdmx"]            # ["sdmx"] или ["sdmx", "excel"]

# Сузить выборку, чтобы SDMX не был гигантским. Совпадение по ПОДСТРОКЕ в названии.
# "*" или отсутствие ключа = взять все значения поля. Примеры названий полей смотри в dry-run.
FILTERS   = {}                  # напр.: {"Год": "2023", "Период": "январь"}

TIMEOUT   = 180                 # таймаут запроса, сек (сайт часто лагает)
INSECURE  = False               # True — не проверять SSL-сертификат (если ругается на сертификат)


## 2. Импорт и подготовка

In [5]:
import os, sys, types, importlib

# гарантируем, что fedstat_capture.py найдётся из корня проекта,
# даже если ноутбук открыт из папки notebooks/
here = os.getcwd()
root = here if os.path.exists(os.path.join(here, "fedstat_capture.py")) else os.path.dirname(here)
if root not in sys.path:
    sys.path.insert(0, root)
os.chdir(root)   # писать fixtures/ в корень проекта

import requests
import fedstat_capture as fc
importlib.reload(fc)   # подхватить правки модуля без перезапуска ядра

session = requests.Session()
session.headers.update(fc.DEFAULT_HEADERS)
os.makedirs(OUT_DIR, exist_ok=True)

def make_args(dry_run):
    return types.SimpleNamespace(
        out=OUT_DIR, format=FORMATS, filters=FILTERS,
        dry_run=dry_run, timeout=TIMEOUT, insecure=INSECURE,
    )

print("Рабочая папка:", os.getcwd())
print("Готово. Индикаторы:", INDICATORS, "| форматы:", FORMATS, "| фильтры:", FILTERS or "нет")


Рабочая папка: /Users/tryadovoi/projects/project-009-fedstat-api
Готово. Индикаторы: ['31452'] | форматы: ['sdmx'] | фильтры: нет


## 3. Осмотр (dry-run) — HTML + список фильтров, без скачивания данных

Безопасно и быстро. Проверь, что для каждого индикатора распарсились поля-фильтры и найден CSRF-токен. Названия полей отсюда можно подставить в `FILTERS` выше.

In [7]:
for ind in INDICATORS:
    fc.capture(str(ind).strip(), make_args(dry_run=True), session)



=== indicator 31452 ===
  [ok] HTML сохранён: fixtures/31452_indicator.html  (HTTP 200, 135511 байт, 1.1с)
  [ok] распарсено полей-фильтров: 7 (всего значений: 154)
        - 'Показатель': 1 знач., тип filterObjectIds, id 0
        - 'Год': 27 знач., тип columnObjectIds, id 3
        - 'Классификатор объектов административно-территориального деления (ОКАТО)': 114 знач., тип lineObjectIds, id 57831
        - 'Единица измерения': 1 знач., тип lineObjectIds, id 30611
        - 'Период': 4 знач., тип columnObjectIds, id 33560
        - 'Рынок жилья': 2 знач., тип lineObjectIds, id 63148
        - 'Типы квартир': 5 знач., тип lineObjectIds, id 58849
  [ok] CSRF-токен: найден
  [dry-run] скачивание данных пропущено. Смотри список полей выше и *_data_ids.json,
            затем при желании сузь выборку через --filter "Поле=значение".


## 4. Захват данных (SDMX/Excel)

Запускай, когда dry-run прошёл успешно. Если без `FILTERS` объём слишком большой или запрос висит — сузь выборку в ячейке настроек и перезапусти ячейки 1–2, затем эту.

In [8]:
for ind in INDICATORS:
    fc.capture(str(ind).strip(), make_args(dry_run=False), session)



=== indicator 31452 ===
  [ok] HTML сохранён: fixtures/31452_indicator.html  (HTTP 200, 135511 байт, 0.7с)
  [ok] распарсено полей-фильтров: 7 (всего значений: 154)
        - 'Показатель': 1 знач., тип filterObjectIds, id 0
        - 'Год': 27 знач., тип columnObjectIds, id 3
        - 'Классификатор объектов административно-территориального деления (ОКАТО)': 114 знач., тип lineObjectIds, id 57831
        - 'Единица измерения': 1 знач., тип lineObjectIds, id 30611
        - 'Период': 4 знач., тип columnObjectIds, id 33560
        - 'Рынок жилья': 2 знач., тип lineObjectIds, id 63148
        - 'Типы квартир': 5 знач., тип lineObjectIds, id 58849
  [ok] CSRF-токен: найден
  [ok] sdmx: fixtures/31452_data.sdmx.xml  (HTTP 200, text/xml, 39718519 байт, 3.9с)


## 5. Запаковать фикстуры для отправки

In [9]:
import shutil, glob
print("Файлы в fixtures/:")
for p in sorted(glob.glob(os.path.join(OUT_DIR, "*"))):
    print("  ", os.path.basename(p), os.path.getsize(p), "байт")

archive = shutil.make_archive("fedstat_fixtures", "zip", OUT_DIR)
print("\nАрхив готов:", os.path.abspath(archive))
print("Пришли этот .zip — там HTML + *_data.sdmx.xml + *_data_ids.json + *_meta.json.")


Файлы в fixtures/:
   31452_data.sdmx.xml 39718519 байт
   31452_data_ids.json 49907 байт
   31452_indicator.html 135511 байт
   31452_meta.json 1281 байт

Архив готов: /Users/tryadovoi/projects/project-009-fedstat-api/fedstat_fixtures.zip
Пришли этот .zip — там HTML + *_data.sdmx.xml + *_data_ids.json + *_meta.json.


---
### Если что-то пошло не так
- **403 (Forbidden)** — антибот. Скажи мне, добавлю ретраи с паузами / подкручу заголовки.
- **Таймаут / висит** — сайт лагает; повтори позже или сузь `FILTERS`.
- **CSRF не найден** или **фильтры не распарсились** — всё равно пришли сохранённый HTML (`fixtures/{id}_indicator.html`): по нему я доработаю парсер.
- **POST вернул HTML вместо данных** — сырой ответ всё равно сохранён в `*_data.sdmx.xml`, загляни в начало файла (часто это страница с ошибкой или 302).